# Yield Surface — Strike × Expiry Heatmap

Pro Symbol eine Heatmap:
- **X-Achse** — Expiration Date
- **Y-Achse** — Strike (absteigend = Standard Options-Chain)
- **Farbe** — Annualisierte Rendite (weiß → dunkelgrün)
- **Rahmenlinie** — Bewertung: grün = Attraktiv · rot gestrichelt = Vorsicht · kein Rahmen = Mittel

## 0 — Setup & Parameter

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns

from src.store import ScanStore

# ── Parameter ────────────────────────────────────────────────────────────────
DB_PATH   = PROJECT_ROOT / 'data' / 'csp_history.duckdb'
RUN_ID    = None    # None = letzter Run  |  z.B. '20260420_151256'
SAVE_PNG  = False   # True = speichert PNGs in output/charts/
# ─────────────────────────────────────────────────────────────────────────────

store = ScanStore(DB_PATH)

# Run-ID auflösen
if RUN_ID is None:
    RUN_ID = store.query(
        "SELECT run_id FROM scan_runs ORDER BY run_ts DESC LIMIT 1"
    ).df().iloc[0]['run_id']

run_ts = store.query(
    f"SELECT run_ts FROM scan_runs WHERE run_id = '{RUN_ID}'"
).df().iloc[0]['run_ts']

print(f'Run : {RUN_ID}')
print(f'Zeit: {run_ts}')

## 1 — Daten laden

In [ ]:
df = store.query(f"""
    SELECT
        symbol,
        expiry_date,
        dte,
        strike,
        ann_yield,
        total_yield,
        score,
        rating,
        delta,
        iv,
        open_interest,
        breakeven
    FROM scan_candidates
    WHERE run_id = '{RUN_ID}'
    ORDER BY symbol, expiry_date, strike DESC
""").df()

symbols = sorted(df['symbol'].unique())
print(f'{len(df)} Kandidaten  |  Symbole: {symbols}')

## 2 — Yield Surface Heatmap

In [ ]:
# ── Farbpalette: weiß → sattes Grün ──────────────────────────────────────────
from matplotlib.colors import LinearSegmentedColormap
CMAP = LinearSegmentedColormap.from_list(
    'yield_green', ['#FFFFFF', '#C7E9C0', '#74C476', '#238B45', '#00441B']
)

BORDER = {
    'Attraktiv': dict(edgecolor='#1a7a2e', lw=2.8,  linestyle='-'),
    'Mittel':    dict(edgecolor='#888888', lw=0.8,  linestyle='-'),
    'Vorsicht':  dict(edgecolor='#c0392b', lw=2.0,  linestyle='--'),
}

if SAVE_PNG:
    chart_dir = PROJECT_ROOT / 'output' / 'charts'
    chart_dir.mkdir(parents=True, exist_ok=True)

for sym in symbols:
    sub = df[df['symbol'] == sym].copy()
    sub['expiry_label'] = pd.to_datetime(sub['expiry_date']).dt.strftime('%b-%y')

    # ── Pivots ────────────────────────────────────────────────────────────────
    pivot_yield = sub.pivot_table(
        index='strike', columns='expiry_label',
        values='ann_yield', aggfunc='max'
    ).sort_index(ascending=False)   # höchster Strike oben

    pivot_score = sub.pivot_table(
        index='strike', columns='expiry_label',
        values='score', aggfunc='max'
    ).reindex_like(pivot_yield)

    pivot_rating = sub.pivot_table(
        index='strike', columns='expiry_label',
        values='rating', aggfunc='first'
    ).reindex_like(pivot_yield)

    n_exp = len(pivot_yield.columns)
    n_str = len(pivot_yield.index)

    cell_w, cell_h = 1.55, 1.1
    fig_w = max(10, n_exp * cell_w + 2.5)
    fig_h = max(4,  n_str * cell_h + 2.5)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    # ── Annot-Matrix: "X.X%\nScore NN" ───────────────────────────────────────
    annot = pd.DataFrame('', index=pivot_yield.index, columns=pivot_yield.columns)
    for r in pivot_yield.index:
        for c in pivot_yield.columns:
            y = pivot_yield.loc[r, c]
            s = pivot_score.loc[r, c]
            if pd.notna(y):
                annot.loc[r, c] = f"{y*100:.1f}%\n▸ {int(s)}"

    # ── Heatmap ───────────────────────────────────────────────────────────────
    vmin = df['ann_yield'].min() * 0.95
    vmax = df['ann_yield'].max() * 1.02

    sns.heatmap(
        pivot_yield,
        ax=ax,
        cmap=CMAP,
        vmin=vmin,
        vmax=vmax,
        annot=annot,
        fmt='',
        annot_kws={'size': 9, 'va': 'center'},
        linewidths=0.5,
        linecolor='#E0E0E0',
        mask=pivot_yield.isna(),
        cbar_kws={'label': 'Ann. Rendite', 'shrink': 0.8, 'format': '{x:.0%}'},
    )

    # ── Leere Zellen grau einfärben ───────────────────────────────────────────
    ax.set_facecolor('#F5F5F5')

    # ── Rating-Rahmen pro Zelle ───────────────────────────────────────────────
    for r_idx, strike in enumerate(pivot_yield.index):
        for c_idx, expiry in enumerate(pivot_yield.columns):
            if pd.isna(pivot_yield.loc[strike, expiry]):
                continue
            rating = pivot_rating.loc[strike, expiry]
            style  = BORDER.get(rating, BORDER['Mittel'])
            ax.add_patch(mpatches.Rectangle(
                (c_idx + 0.06, r_idx + 0.06), 0.88, 0.88,
                fill=False, zorder=4, **style
            ))

    # ── Achsen & Titel ────────────────────────────────────────────────────────
    ax.set_title(
        f'{sym}  —  Yield Surface   ({RUN_ID})',
        fontsize=14, fontweight='bold', pad=14,
    )
    ax.set_xlabel('Expiration Date', fontsize=11, labelpad=8)
    ax.set_ylabel('Strike', fontsize=11, labelpad=8)
    ax.tick_params(axis='x', rotation=35, labelsize=9)
    ax.tick_params(axis='y', rotation=0,  labelsize=9)

    # ── Legende ───────────────────────────────────────────────────────────────
    legend_handles = [
        mpatches.Patch(facecolor='white', edgecolor='#1a7a2e', lw=2.5, label='Attraktiv  (Score ≥ 70)'),
        mpatches.Patch(facecolor='white', edgecolor='#888888', lw=0.8, label='Mittel     (Score 45–69)'),
        mpatches.Patch(facecolor='white', edgecolor='#c0392b', lw=2.0,
                       linestyle='--', label='Vorsicht   (Score < 45)'),
    ]
    ax.legend(
        handles=legend_handles,
        loc='upper left', bbox_to_anchor=(1.18, 1.0),
        fontsize=8.5, title='Bewertung', title_fontsize=9,
        framealpha=0.9,
    )

    plt.tight_layout()

    if SAVE_PNG:
        out = chart_dir / f'yield_surface_{sym}_{RUN_ID}.png'
        fig.savefig(out, dpi=150, bbox_inches='tight')
        print(f'Gespeichert: {out}')

    plt.show()

## 3 — Score-Profil & Delta-Übersicht

In [ ]:
# Score-Profil: für jeden Strike den Score über alle Expiries
# Delta als Sekundärachse — zeigt die Zuweisung-Wahrscheinlichkeit

for sym in symbols:
    sub = df[df['symbol'] == sym].copy()
    sub['expiry_label'] = pd.to_datetime(sub['expiry_date']).dt.strftime('%b-%y')
    strikes = sorted(sub['strike'].unique(), reverse=True)

    fig, axes = plt.subplots(
        1, len(strikes),
        figsize=(len(strikes) * 3.8, 4.5),
        sharey=False,
    )
    if len(strikes) == 1:
        axes = [axes]

    fig.suptitle(
        f'{sym}  —  Score & Delta pro Strike',
        fontsize=13, fontweight='bold', y=1.01,
    )

    COLOR_RATING = {'Attraktiv': '#27ae60', 'Mittel': '#f39c12', 'Vorsicht': '#e74c3c'}

    for ax, strike in zip(axes, strikes):
        grp = sub[sub['strike'] == strike].sort_values('expiry_date')

        bar_colors = [COLOR_RATING.get(r, '#888888') for r in grp['rating']]
        bars = ax.bar(grp['expiry_label'], grp['score'],
                      color=bar_colors, alpha=0.85, edgecolor='white', linewidth=0.6)

        ax.set_ylim(0, 100)
        ax.axhline(70, color='#27ae60', lw=1.2, linestyle=':', alpha=0.7)
        ax.axhline(45, color='#e74c3c', lw=1.2, linestyle=':', alpha=0.7)

        # Score-Wert über jedem Balken
        for bar, score in zip(bars, grp['score']):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
                    str(int(score)), ha='center', va='bottom', fontsize=7.5)

        # Delta-Linie auf Sekundärachse (als Proxy für Assignment-Wahrscheinlichkeit)
        if grp['delta'].notna().any():
            ax2 = ax.twinx()
            ax2.plot(grp['expiry_label'], grp['delta'].abs(),
                     color='#2c3e50', lw=1.5, marker='o', markersize=4,
                     linestyle='--', alpha=0.7)
            ax2.set_ylim(0, 0.5)
            ax2.set_ylabel('|Delta|', fontsize=7, color='#2c3e50')
            ax2.tick_params(axis='y', labelsize=7, labelcolor='#2c3e50')
        
        ax.set_title(f'Strike {strike:.0f}', fontsize=10, fontweight='bold')
        ax.set_ylabel('Score', fontsize=8)
        ax.tick_params(axis='x', rotation=40, labelsize=7)
        ax.tick_params(axis='y', labelsize=8)
        ax.set_xlabel('')

    plt.tight_layout()

    if SAVE_PNG:
        out = chart_dir / f'score_profile_{sym}_{RUN_ID}.png'
        fig.savefig(out, dpi=150, bbox_inches='tight')
        print(f'Gespeichert: {out}')

    plt.show()

## 4 — Rendite vs. Downside-Puffer (Effizienz-Scatter)

In [ ]:
# Klassische Risk/Reward-Darstellung:
# X = Downside-Puffer (Sicherheit), Y = Ann. Rendite (Ertrag)
# Optimale Kandidaten: oben rechts  |  Ineffizient: unten links

fig, axes = plt.subplots(1, len(symbols),
                         figsize=(7 * len(symbols), 5.5),
                         squeeze=False)

COLOR_RATING = {'Attraktiv': '#27ae60', 'Mittel': '#f39c12', 'Vorsicht': '#e74c3c'}

for ax, sym in zip(axes[0], symbols):
    sub = df[df['symbol'] == sym].copy()
    sub['cushion'] = (sub['breakeven'].apply(
        lambda be: (sub.loc[sub.index[0], 'breakeven'])) * 0  # placeholder
    )
    # Downside-Puffer = (Spot - Strike) / Spot — aus Breakeven ableitbar
    # Wir berechnen direkt: jede Zeile hat Breakeven und kann Spot zurückrechnen
    # Fallback: nutze score als Proxy

    # Für sauberen Puffer: aus DuckDB holen
    sub2 = store.query(f"""
        SELECT strike, expiry_date, ann_yield, score, rating,
               ROUND((spot - strike) / spot * 100, 2) AS cushion_pct,
               dte, delta
        FROM scan_candidates
        WHERE run_id = '{RUN_ID}' AND symbol = '{sym}'
    """).df()

    for rating, group in sub2.groupby('rating'):
        color = COLOR_RATING.get(rating, '#888888')
        sc = ax.scatter(
            group['cushion_pct'],
            group['ann_yield'] * 100,
            c=color,
            s=group['score'] * 2.5,
            alpha=0.82,
            edgecolors='white',
            linewidths=0.6,
            label=rating,
            zorder=3,
        )

    # Label: Strike + Expiry
    for _, row in sub2.iterrows():
        exp_label = pd.to_datetime(row['expiry_date']).strftime('%b-%y')
        ax.annotate(
            f"K{row['strike']:.0f}\n{exp_label}",
            (row['cushion_pct'], row['ann_yield'] * 100),
            fontsize=6.5, alpha=0.8,
            xytext=(5, 4), textcoords='offset points',
        )

    ax.set_title(f'{sym}  —  Effizienz-Scatter',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Downside-Puffer  (Spot − Strike) / Spot  %',  fontsize=10)
    ax.set_ylabel('Ann. Rendite  %', fontsize=10)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
    ax.grid(alpha=0.25)
    ax.legend(title='Bewertung', fontsize=8, title_fontsize=8)

    # Notiz zur Punktgröße
    ax.text(0.98, 0.03, 'Punktgröße = Score',
            transform=ax.transAxes, ha='right', fontsize=7.5,
            color='#666666', style='italic')

plt.tight_layout()

if SAVE_PNG:
    out = chart_dir / f'efficiency_scatter_{RUN_ID}.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    print(f'Gespeichert: {out}')

plt.show()

In [ ]:
store.close()